# Week 1: Colab LLM/SLM Pipeline Smoke Test

This notebook is the first practical step for the unlearning thesis project. It loads a small instruction-tuned causal language model on Google Colab, verifies GPU access, and generates answers from prompts.

Recommended Colab runtime: **T4 GPU**.

Default model: `Qwen/Qwen2.5-0.5B-Instruct`

Good fallback model: `HuggingFaceTB/SmolLM2-360M-Instruct`

Later, this same pipeline can be extended into:

- synthetic fact dataset creation
- baseline fine-tuning
- LoRA/QLoRA adapter training
- forget/retain evaluation
- unlearning experiments


## 0. Runtime Setup

In Colab, go to:

`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`

Then run the cells from top to bottom.

In [ ]:
# Install the main libraries needed for inference now and fine-tuning later.
!pip -q install -U transformers accelerate bitsandbytes peft datasets sentencepiece huggingface_hub pandas

## 1. Check GPU and Libraries

In [ ]:
import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU memory: {total_gb:.2f} GB')
else:
    print('No GPU found. This notebook can run on CPU for tiny tests, but it will be slow.')

## 2. Choose a Small Instruction Model

For the thesis, start small. You need repeatable runs more than you need a huge model.

Use `Qwen/Qwen2.5-0.5B-Instruct` first. If memory is tight or loading is slow, switch to `HuggingFaceTB/SmolLM2-360M-Instruct`.

In [ ]:
# Main choice for Week 1.
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

# Smaller fallback if needed.
# MODEL_ID = 'HuggingFaceTB/SmolLM2-360M-Instruct'

# A 0.5B model usually fits on a T4 in float16. Keep 4-bit off for the first smoke test.
# Turn this on later if you test larger models or want lower memory use.
USE_4BIT = False

print('Selected model:', MODEL_ID)
print('Use 4-bit loading:', USE_4BIT)

## 3. Load Tokenizer and Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map='auto',
    )
else:
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map='auto' if torch.cuda.is_available() else None,
    )

model.eval()

print('Loaded model:', MODEL_ID)
print('Model device:', next(model.parameters()).device)
print('Parameter dtype:', next(model.parameters()).dtype)

## 4. Generate Answers

This helper uses the model's chat template when available. That makes it work well with instruction-tuned models such as Qwen and SmolLM.

In [ ]:
@torch.inference_mode()
def generate_answer(
    user_prompt,
    system_prompt='You are a concise and helpful research assistant.',
    max_new_tokens=160,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]

    if getattr(tokenizer, 'chat_template', None):
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        text = f'System: {system_prompt}\nUser: {user_prompt}\nAssistant:'

    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=do_sample,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    new_tokens = output_ids[0][inputs['input_ids'].shape[-1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer


test_prompt = 'In two short paragraphs, explain what machine unlearning means for language models.'
print(generate_answer(test_prompt))

## 5. Try Thesis-Relevant Prompts

In [ ]:
prompts = [
    'Define targeted forgetting in machine unlearning using one simple example.',
    'What is the difference between a forget set and a retain set?',
    'Give me three evaluation metrics for an LLM unlearning experiment.',
    'Why is a small language model useful for a master thesis experiment in Google Colab?',
]

for i, prompt in enumerate(prompts, start=1):
    print('=' * 80)
    print(f'Prompt {i}: {prompt}')
    print('-' * 80)
    print(generate_answer(prompt, max_new_tokens=140))


## 6. Batch Generation Table

This format will be useful later when you evaluate forget-set and retain-set prompts.

In [ ]:
rows = []

for prompt in prompts:
    answer = generate_answer(prompt, max_new_tokens=120, temperature=0.3, top_p=0.9)
    rows.append({
        'model_id': MODEL_ID,
        'prompt': prompt,
        'answer': answer,
    })

df = pd.DataFrame(rows)
df

## 7. Save Generated Answers

Saving outputs from the start is a good research habit. Later, every experiment should save prompts, outputs, model name, seed, method, and timestamp.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Files are saved in the organized Week 1 results folder.
output_dir = Path('/content/drive/MyDrive/Thesis/Week 1/results')
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / 'week1_generated_answers.csv'
jsonl_path = output_dir / 'week1_generated_answers.jsonl'

df.to_csv(csv_path, index=False)
with jsonl_path.open('w', encoding='utf-8') as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('Saved CSV:', csv_path)
print('Saved JSONL:', jsonl_path)

## 8. Your Own Prompt

Change `my_prompt` and run the cell.

In [ ]:
my_prompt = 'Give me a simple experimental plan for testing unlearning on synthetic facts.'

answer = generate_answer(my_prompt, max_new_tokens=220, temperature=0.5)
print(answer)

## 9. Optional: Clear GPU Memory

Run this only when you are done with the model or want to load a different one.

In [ ]:
# Optional cleanup cell. Uncomment if you need to free memory.
# del model
# del tokenizer
# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
# print('Cleaned up model memory.')

## Next Step

Once this notebook runs successfully, the next notebook should add a small synthetic factual dataset:

- create 100 fictional identities
- generate 3 to 5 facts per identity
- split identities into forget and retain sets
- ask the base model questions before fine-tuning
- save baseline outputs

That will become the foundation for baseline fine-tuning and unlearning.